# Simplify A/B/C Test — Slide Analysis Notebook

Runs every analysis needed for the presentation and report.
Each section corresponds to one slide. Outputs are pandas DataFrames you can screenshot or copy into your deck.

**Prerequisites:** run `01_data_prep.py` and `02_gen_llm_data.py` first so that `data/human_wide.csv`, `data/human_long.csv`, `data/llm_wide.csv`, `data/llm_long.csv` exist.

## 0. Setup

In [5]:
import os, warnings
import numpy as np
import pandas as pd
from scipy import stats

import statsmodels.stats.power as smp

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)

DATA_DIR = os.path.join(os.getcwd(), 'data')
OUTCOMES = ['signup', 'useful', 'regular', 'recommend']
ALPHA    = 0.05

df_human_wide = pd.read_csv(os.path.join(DATA_DIR, 'human_wide.csv'))
df_human_long = pd.read_csv(os.path.join(DATA_DIR, 'human_long.csv'))
df_llm_wide   = pd.read_csv(os.path.join(DATA_DIR, 'llm_wide.csv'))
df_llm_long   = pd.read_csv(os.path.join(DATA_DIR, 'llm_long.csv'))
df_comb_long  = pd.concat([df_human_long, df_llm_long], ignore_index=True)

print(f'Human : {len(df_human_wide):>5,} respondents   |  {len(df_human_long):>5,} observations')
print(f'LLM   : {len(df_llm_wide):>5,} respondents   |  {len(df_llm_long):>5,} observations')
print(f'Total : {len(df_human_wide)+len(df_llm_wide):>5,} respondents   |  {len(df_human_long)+len(df_llm_long):>5,} observations')

Human :    49 respondents   |     98 observations
LLM   : 1,056 respondents   |  2,112 observations
Total : 1,105 respondents   |  2,210 observations


## Slide 1 — Data Overview

In [6]:
def data_overview(df_human, df_llm):
    h_ab, h_ac = int((df_human.group=='AB').sum()), int((df_human.group=='AC').sum())
    l_ab, l_ac = int((df_llm.group  =='AB').sum()), int((df_llm.group  =='AC').sum())
    rows = [
        ('Control + Treatment B (AB group)',                     h_ab,  l_ab,  h_ab+l_ab),
        ('Control + Treatment C (AC group)',                     h_ac,  l_ac,  h_ac+l_ac),
        ('Total respondents',                                    h_ab+h_ac, l_ab+l_ac, h_ab+h_ac+l_ab+l_ac),
        ('Total observations (within-subjects, 2 pages/person)', 2*(h_ab+h_ac), 2*(l_ab+l_ac), 2*(h_ab+h_ac+l_ab+l_ac)),
    ]
    return pd.DataFrame(rows, columns=['', 'Human', 'LLM (GPT)', 'Combined']).set_index('')

overview = data_overview(df_human_wide, df_llm_wide)
overview

,Human,LLM (GPT),Combined
,,,
Control + Treatment B (AB group),25,533,558
Control + Treatment C (AC group),24,523,547
Total respondents,49,1056,1105
"Total observations (within-subjects, 2 pages/person)",98,2112,2210


## Slide 2 — Summary Statistics

Mean ± std of each 0–5 outcome, by page, split by data source.

In [7]:
def summary_table(df_long, label):
    out = []
    for page in sorted(df_long['page'].unique()):
        sub = df_long[df_long.page == page]
        row = {'Source': label, 'Page': page, 'n': len(sub)}
        for o in OUTCOMES:
            row[f'{o}'] = f'{sub[o].mean():.2f} \u00b1 {sub[o].std():.2f}'
        out.append(row)
    return pd.DataFrame(out)

summary_df = pd.concat([
    summary_table(df_human_long, 'Human'),
    summary_table(df_llm_long,   'LLM'),
    summary_table(df_comb_long,  'Combined'),
], ignore_index=True)
summary_df

,Source,Page,n,signup,useful,regular,recommend
0,Human,A,49,3.22 ± 1.26,2.74 ± 0.97,3.16 ± 1.28,3.02 ± 1.13
1,Human,B,25,3.24 ± 0.97,3.12 ± 0.97,3.32 ± 1.07,3.16 ± 1.14
2,Human,C,24,2.96 ± 1.30,2.87 ± 0.92,3.12 ± 1.12,2.83 ± 1.17
3,LLM,A,1056,3.61 ± 0.86,2.85 ± 0.62,3.50 ± 0.94,3.32 ± 0.82
4,LLM,B,533,3.49 ± 0.80,3.14 ± 0.81,3.50 ± 0.85,3.43 ± 0.82
5,LLM,C,523,3.29 ± 0.85,2.92 ± 0.67,3.29 ± 0.82,3.25 ± 0.82
6,Combined,A,1105,3.60 ± 0.89,2.84 ± 0.64,3.49 ± 0.96,3.30 ± 0.84
7,Combined,B,558,3.48 ± 0.81,3.14 ± 0.82,3.49 ± 0.86,3.41 ± 0.84
8,Combined,C,547,3.28 ± 0.87,2.92 ± 0.68,3.28 ± 0.83,3.23 ± 0.84


### Slide 2a — Raw Outcome Means by Treatment × Sample (deck format)

One row per Page × Sample, just the mean of each outcome (0–5 scale).

In [15]:
# Slide-ready: Raw Outcome Means — Page x Sample
PAGE_LABEL = {'A': 'A (control)', 'B': 'B (outcome)', 'C': 'C (efficiency)'}
OUTCOME_LABEL = {'signup': 'Signup', 'useful': 'Usefulness',
                 'regular': 'Regular use', 'recommend': 'Recommend'}

rows = []
for page in ['A', 'B', 'C']:
    for src_label, df in [('Human', df_human_long), ('LLM', df_llm_long), ('Combined', df_comb_long)]:
        sub = df[df.page == page]
        if len(sub) == 0:
            continue
        row = {'Page': PAGE_LABEL[page], 'Sample': src_label, 'n': len(sub)}
        for o in OUTCOMES:
            row[OUTCOME_LABEL[o]] = round(sub[o].mean(), 2)
        rows.append(row)

raw_means = pd.DataFrame(rows)
# Blank out repeated Page labels so the slide table reads top-down like the deck
raw_means_display = raw_means.copy()
raw_means_display.loc[raw_means_display.duplicated(subset=['Page']), 'Page'] = ''
print(raw_means_display.to_string(index=False))
raw_means_display

          Page   Sample    n  Signup  Usefulness  Regular use  Recommend
   A (control)    Human   49    3.22        2.74         3.16       3.02
                    LLM 1056    3.61        2.85         3.50       3.32
               Combined 1105    3.60        2.84         3.49       3.30
   B (outcome)    Human   25    3.24        3.12         3.32       3.16
                    LLM  533    3.49        3.14         3.50       3.43
               Combined  558    3.48        3.14         3.49       3.41
C (efficiency)    Human   24    2.96        2.87         3.12       2.83
                    LLM  523    3.29        2.92         3.29       3.25
               Combined  547    3.28        2.92         3.28       3.23


,Page,Sample,n,Signup,Usefulness,Regular use,Recommend
0,A (control),Human,49,3.22,2.74,3.16,3.02
1,,LLM,1056,3.61,2.85,3.50,3.32
2,,Combined,1105,3.60,2.84,3.49,3.30
3,B (outcome),Human,25,3.24,3.12,3.32,3.16
4,,LLM,533,3.49,3.14,3.50,3.43
5,,Combined,558,3.48,3.14,3.49,3.41
6,C (efficiency),Human,24,2.96,2.87,3.12,2.83
7,,LLM,523,3.29,2.92,3.29,3.25
8,,Combined,547,3.28,2.92,3.28,3.23


In [8]:
print('Demographics (human respondents):')
df_h_pers = df_human_long[df_human_long.page == 'A']
for col in ['gender', 'field', 'job_status', 'used_ai', 'heard_simplify']:
    print(f'\n-- {col} --')
    print(df_h_pers[col].value_counts().to_string())

Demographics (human respondents):

-- gender --
gender
Female    34
Male      15

-- field --
field
Information Systems / Data Science    33
Computer Science / Engineering         7
Business                               3
Design                                 2
Natural Sciences                       1
Data Analytic for Science              1
Social Sciences                        1
Arts                                   1

-- job_status --
job_status
Actively applying for internships               21
Actively applying for full-time jobs            11
Not currently looking for jobs                  10
Exploring opportunities but not applying yet     7

-- used_ai --
used_ai
Yes    40
No      9

-- heard_simplify --
heard_simplify
Yes         33
No          12
Not sure     4


## Slide 3 — Balance Check (AB vs AC on pre-treatment covariates)

Good randomization → p-values > 0.05.

### Slide 3a — Balance Check (Human only, deck format)

Exact format of the "Balance Check: Human Sample" slide: one-row-per-covariate with `AB (n=25)` / `AC (n=24)` / `p-value` columns.

In [14]:
# Slide-ready: Balance Check — Human Sample only
# Matches the format on the deck: Covariate | AB (n=..) | AC (n=..) | p-value
df_h = df_human_long[df_human_long.page == 'A'].copy()
ab, ac = df_h[df_h.group == 'AB'], df_h[df_h.group == 'AC']

def _prop(a, b, val):
    a, b = a.dropna(), b.dropna()
    n1, n2 = len(a), len(b)
    x1, x2 = (a == val).sum(), (b == val).sum()
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z  = (x1/n1 - x2/n2) / se if se > 0 else 0
    p  = 2 * stats.norm.sf(abs(z))
    return x1/n1*100, x2/n2*100, p

ab_age = ab['age'].dropna().astype(float)
ac_age = ac['age'].dropna().astype(float)
_, p_age = stats.ttest_ind(ab_age, ac_age, equal_var=False)

female_ab, female_ac, p_f   = _prop(ab['gender'],         ac['gender'],         'Female')
ai_ab,     ai_ac,     p_ai  = _prop(ab['used_ai'],        ac['used_ai'],        'Yes')
heard_ab,  heard_ac,  p_h   = _prop(ab['heard_simplify'], ac['heard_simplify'], 'Yes')

balance_slide = pd.DataFrame([
    {'Covariate': 'Mean age (years)',       f'AB (n={len(ab)})': f'{ab_age.mean():.1f}',  f'AC (n={len(ac)})': f'{ac_age.mean():.1f}',  'p-value': round(p_age, 2)},
    {'Covariate': 'Female',                 f'AB (n={len(ab)})': f'{female_ab:.1f}%',     f'AC (n={len(ac)})': f'{female_ac:.1f}%',     'p-value': round(p_f,   2)},
    {'Covariate': 'Used AI for job search', f'AB (n={len(ab)})': f'{ai_ab:.1f}%',         f'AC (n={len(ac)})': f'{ai_ac:.1f}%',         'p-value': round(p_ai,  2)},
    {'Covariate': 'Heard of Simplify before', f'AB (n={len(ab)})': f'{heard_ab:.1f}%',    f'AC (n={len(ac)})': f'{heard_ac:.1f}%',      'p-value': round(p_h,   2)},
])

print('Balance Check: Human Sample')
print('A: Control Group, B: Treatment One (outcome-focus), C: Treatment Two (efficiency-focus)\n')
print(balance_slide.to_string(index=False))
print(f"\nAll p > 0.05 → groups are balanced across pre-treatment covariates.")
balance_slide

Balance Check: Human Sample
A: Control Group, B: Treatment One (outcome-focus), C: Treatment Two (efficiency-focus)

               Covariate AB (n=25) AC (n=24)  p-value
        Mean age (years)      24.7      25.2     0.66
                  Female     64.0%     75.0%     0.40
  Used AI for job search     80.0%     83.3%     0.76
Heard of Simplify before     72.0%     62.5%     0.48

All p > 0.05 → groups are balanced across pre-treatment covariates.


,Covariate,AB (n=25),AC (n=24),p-value
0,Mean age (years),24.7,25.2,0.66
1,Female,64.0%,75.0%,0.40
2,Used AI for job search,80.0%,83.3%,0.76
3,Heard of Simplify before,72.0%,62.5%,0.48


### Slide 4a — ATE: B vs A (deck format)

Within-subject paired means for the AB group. Three tables: Human, LLM, Combined.
Columns: Outcome | Control A mean | Treatment B mean | Difference | % lift | p

In [16]:
# Slide-ready: ATE B vs A — Outcome | Control A mean | Treatment B mean | Diff | % lift | p
OUTCOMES_ORDERED = ['useful', 'regular', 'recommend', 'signup']
OUTCOME_LABEL = {'useful': 'Usefulness', 'regular': 'Regular use',
                 'recommend': 'Recommend', 'signup': 'Signup'}

def _fmt_p(p):
    if np.isnan(p):  return 'NA'
    if p < 0.001:    return '<.001'
    if p < 0.01:     return f'{p:.3f}'.lstrip('0')   # e.g., .003
    if p < 1:        return f'{p:.2f}'.lstrip('0')   # e.g., .21, .50
    return f'{p:.2f}'                                 # 1.00

def ate_b_vs_a(df_wide, label):
    g = df_wide[df_wide.group == 'AB'].copy()
    rows = []
    for o in OUTCOMES_ORDERED:
        paired = g[[f'A_{o}', f'B_{o}']].dropna().astype(float)
        a_mean = paired[f'A_{o}'].mean()
        b_mean = paired[f'B_{o}'].mean()
        diff   = b_mean - a_mean
        lift   = diff / a_mean * 100 if a_mean != 0 else 0
        _, p   = stats.ttest_rel(paired[f'B_{o}'], paired[f'A_{o}'])
        rows.append({
            'Outcome':          OUTCOME_LABEL[o],
            'Control A mean':   round(a_mean, 2),
            'Treatment B mean': round(b_mean, 2),
            'Difference':       f'{diff:+.2f}',
            '% lift':           f'{lift:+.0f}%',
            'p':                _fmt_p(p),
        })
    df = pd.DataFrame(rows)
    print(f'\n=== {label}: Page B vs Page A ===')
    print(df.to_string(index=False))
    return df

ate_b_human    = ate_b_vs_a(df_human_wide, f'HUMAN (n={(df_human_wide.group=="AB").sum()})')
ate_b_llm      = ate_b_vs_a(df_llm_wide,   f'LLM (n={(df_llm_wide.group=="AB").sum()})')
ate_b_combined = ate_b_vs_a(pd.concat([df_human_wide, df_llm_wide], ignore_index=True),
                            f'COMBINED (n={(df_human_wide.group=="AB").sum()+(df_llm_wide.group=="AB").sum()})')

ate_b_human.to_csv(os.path.join(DATA_DIR, 'slide_ate_B_vs_A_human.csv'),    index=False)
ate_b_llm.to_csv(os.path.join(DATA_DIR,   'slide_ate_B_vs_A_llm.csv'),      index=False)
ate_b_combined.to_csv(os.path.join(DATA_DIR, 'slide_ate_B_vs_A_combined.csv'), index=False)


=== HUMAN (n=25): Page B vs Page A ===
    Outcome  Control A mean  Treatment B mean Difference % lift    p
 Usefulness            2.62              3.12      +0.50   +19% .001
Regular use            3.08              3.32      +0.24    +8%  .21
  Recommend            3.04              3.16      +0.12    +4%  .50
     Signup            3.24              3.24      +0.00    +0% 1.00

=== LLM (n=533): Page B vs Page A ===
    Outcome  Control A mean  Treatment B mean Difference % lift     p
 Usefulness            2.81              3.14      +0.33   +12% <.001
Regular use            3.34              3.50      +0.16    +5% <.001
  Recommend            3.32              3.43      +0.11    +3% <.001
     Signup            3.53              3.49      -0.04    -1% <.001

=== COMBINED (n=558): Page B vs Page A ===
    Outcome  Control A mean  Treatment B mean Difference % lift     p
 Usefulness            2.80              3.14      +0.34   +12% <.001
Regular use            3.33              3

### Slide 4b — ATE: C vs A (single compact table, Option A layout)

One table combining Human / LLM / Combined. Each cell shows the mean difference (Page C − Page A) and paired t-test p-value.

In [18]:
# Slide-ready consolidated ATE table: C vs A — Human / LLM / Combined side by side
OUTCOMES_ORDERED = ['useful', 'regular', 'recommend', 'signup']
OUTCOME_LABEL = {'useful': 'Usefulness', 'regular': 'Regular use',
                 'recommend': 'Recommend', 'signup': 'Signup'}

def _fmt_p(p):
    if np.isnan(p):  return 'NA'
    if p < 0.001:    return '<.001'
    if p < 0.01:     return f'{p:.3f}'.lstrip('0')
    if p < 1:        return f'{p:.2f}'.lstrip('0')
    return f'{p:.2f}'

def paired_diff(df_wide, group, treat_page, outcome):
    g = df_wide[df_wide.group == group]
    pr = g[[f'A_{outcome}', f'{treat_page}_{outcome}']].dropna().astype(float)
    diff = pr[f'{treat_page}_{outcome}'].mean() - pr[f'A_{outcome}'].mean()
    _, p = stats.ttest_rel(pr[f'{treat_page}_{outcome}'], pr[f'A_{outcome}'])
    return diff, p, len(pr)

def ate_compact(df_human, df_llm, df_comb, group='AC', treat_page='C'):
    sources = [('Human', df_human), ('LLM', df_llm), ('Combined', df_comb)]
    ns = {name: (df.group == group).sum() for name, df in sources}
    cols = {name: f'{name} (n={ns[name]})' for name in ns}

    rows = []
    for o in OUTCOMES_ORDERED:
        row = {'Outcome': OUTCOME_LABEL[o]}
        for name, df in sources:
            d, p, _ = paired_diff(df, group, treat_page, o)
            row[cols[name]] = f'{d:+.2f} (p={_fmt_p(p)})'
        rows.append(row)
    return pd.DataFrame(rows)

ate_c_compact = ate_compact(
    df_human_wide,
    df_llm_wide,
    pd.concat([df_human_wide, df_llm_wide], ignore_index=True),
    group='AC', treat_page='C',
)

print('Average Treatment Effect: A vs C  (paired, within-subjects)')
print('Cells show mean diff (C − A) and paired t-test p-value.\n')
print(ate_c_compact.to_string(index=False))

ate_c_compact.to_csv(os.path.join(DATA_DIR, 'slide_ate_C_vs_A_compact.csv'), index=False)
ate_c_compact

Average Treatment Effect: A vs C  (paired, within-subjects)
Cells show mean diff (C − A) and paired t-test p-value.

    Outcome  Human (n=24)     LLM (n=523) Combined (n=547)
 Usefulness +0.05 (p=.80)  +0.04 (p=.002)   +0.04 (p=.004)
Regular use -0.12 (p=.50) -0.38 (p=<.001)  -0.37 (p=<.001)
  Recommend -0.17 (p=.46) -0.06 (p=<.001)  -0.07 (p=<.001)
     Signup -0.25 (p=.21) -0.41 (p=<.001)  -0.40 (p=<.001)


,Outcome,Human (n=24),LLM (n=523),Combined (n=547)
0,Usefulness,+0.05 (p=.80),+0.04 (p=.002),+0.04 (p=.004)
1,Regular use,-0.12 (p=.50),-0.38 (p=<.001),-0.37 (p=<.001)
2,Recommend,-0.17 (p=.46),-0.06 (p=<.001),-0.07 (p=<.001)
3,Signup,-0.25 (p=.21),-0.41 (p=<.001),-0.40 (p=<.001)


### Slide 4c — ATE: C vs A (detailed tables, one per source)

Same layout as Slide 4a (B vs A): three separate tables (Human / LLM / Combined) with `Outcome | Control A mean | Treatment C mean | Difference | % lift | p` columns.

In [19]:
# Slide-ready: ATE C vs A — Outcome | Control A mean | Treatment C mean | Diff | % lift | p
OUTCOMES_ORDERED = ['useful', 'regular', 'recommend', 'signup']
OUTCOME_LABEL = {'useful': 'Usefulness', 'regular': 'Regular use',
                 'recommend': 'Recommend', 'signup': 'Signup'}

def _fmt_p(p):
    if np.isnan(p):  return 'NA'
    if p < 0.001:    return '<.001'
    if p < 0.01:     return f'{p:.3f}'.lstrip('0')
    if p < 1:        return f'{p:.2f}'.lstrip('0')
    return f'{p:.2f}'

def ate_c_vs_a(df_wide, label):
    g = df_wide[df_wide.group == 'AC'].copy()
    rows = []
    for o in OUTCOMES_ORDERED:
        paired = g[[f'A_{o}', f'C_{o}']].dropna().astype(float)
        a_mean = paired[f'A_{o}'].mean()
        c_mean = paired[f'C_{o}'].mean()
        diff   = c_mean - a_mean
        lift   = diff / a_mean * 100 if a_mean != 0 else 0
        _, p   = stats.ttest_rel(paired[f'C_{o}'], paired[f'A_{o}'])
        rows.append({
            'Outcome':          OUTCOME_LABEL[o],
            'Control A mean':   round(a_mean, 2),
            'Treatment C mean': round(c_mean, 2),
            'Difference':       f'{diff:+.2f}',
            '% lift':           f'{lift:+.0f}%',
            'p':                _fmt_p(p),
        })
    df = pd.DataFrame(rows)
    print(f'\n=== {label}: Page C vs Page A ===')
    print(df.to_string(index=False))
    return df

n_h = (df_human_wide.group == 'AC').sum()
n_l = (df_llm_wide.group   == 'AC').sum()
n_c = n_h + n_l

ate_c_human    = ate_c_vs_a(df_human_wide, f'HUMAN (n={n_h})')
ate_c_llm      = ate_c_vs_a(df_llm_wide,   f'LLM (n={n_l})')
ate_c_combined = ate_c_vs_a(
    pd.concat([df_human_wide, df_llm_wide], ignore_index=True),
    f'COMBINED (n={n_c})',
)

ate_c_human.to_csv(os.path.join(DATA_DIR,    'slide_ate_C_vs_A_human.csv'),    index=False)
ate_c_llm.to_csv(os.path.join(DATA_DIR,      'slide_ate_C_vs_A_llm.csv'),      index=False)
ate_c_combined.to_csv(os.path.join(DATA_DIR, 'slide_ate_C_vs_A_combined.csv'), index=False)


=== HUMAN (n=24): Page C vs Page A ===
    Outcome  Control A mean  Treatment C mean Difference % lift   p
 Usefulness            2.91              2.95      +0.05    +2% .80
Regular use            3.25              3.12      -0.12    -4% .50
  Recommend            3.00              2.83      -0.17    -6% .46
     Signup            3.21              2.96      -0.25    -8% .21

=== LLM (n=523): Page C vs Page A ===
    Outcome  Control A mean  Treatment C mean Difference % lift     p
 Usefulness            2.88              2.92      +0.04    +1%  .002
Regular use            3.67              3.29      -0.38   -10% <.001
  Recommend            3.32              3.25      -0.06    -2% <.001
     Signup            3.70              3.29      -0.41   -11% <.001

=== COMBINED (n=547): Page C vs Page A ===
    Outcome  Control A mean  Treatment C mean Difference % lift     p
 Usefulness            2.88              2.92      +0.04    +1%  .004
Regular use            3.65              3.28  

In [17]:
def balance_check(df_long, label):
    df_pers = df_long[df_long.page == 'A'].copy()
    ab = df_pers[df_pers.group == 'AB']
    ac = df_pers[df_pers.group == 'AC']
    rows = []

    ab_age = ab['age'].dropna().astype(float)
    ac_age = ac['age'].dropna().astype(float)
    if len(ab_age) > 1 and len(ac_age) > 1:
        _, p = stats.ttest_ind(ab_age, ac_age, equal_var=False)
        rows.append({'Source': label, 'Variable': 'age (mean)',
                     'AB': f'{ab_age.mean():.2f}', 'AC': f'{ac_age.mean():.2f}',
                     'p-value': round(p, 3),
                     'Balance?': 'OK' if p > 0.05 else 'IMBALANCED'})

    for col, val in [('gender', 'Female'), ('used_ai', 'Yes'), ('heard_simplify', 'Yes')]:
        ab_v, ac_v = ab[col].dropna(), ac[col].dropna()
        if len(ab_v) == 0 or len(ac_v) == 0:
            continue
        ab_n1, ab_n = (ab_v == val).sum(), len(ab_v)
        ac_n1, ac_n = (ac_v == val).sum(), len(ac_v)
        p_pool = (ab_n1 + ac_n1) / (ab_n + ac_n)
        se = np.sqrt(p_pool * (1-p_pool) * (1/ab_n + 1/ac_n))
        z  = ((ab_n1/ab_n) - (ac_n1/ac_n)) / se if se > 0 else 0
        p  = 2 * stats.norm.sf(abs(z))
        rows.append({'Source': label, 'Variable': f'{col} = {val}',
                     'AB': f'{ab_n1/ab_n*100:.1f}%', 'AC': f'{ac_n1/ac_n*100:.1f}%',
                     'p-value': round(p, 3),
                     'Balance?': 'OK' if p > 0.05 else 'IMBALANCED'})

    for col in ['field', 'job_status', 'difficulty']:
        if col not in df_pers.columns:
            continue
        ct = pd.crosstab(df_pers['group'], df_pers[col])
        if ct.shape[1] < 2:
            continue
        try:
            _, p, _, _ = stats.chi2_contingency(ct)
            rows.append({'Source': label, 'Variable': col + ' (chi2)',
                         'AB': '—', 'AC': '—',
                         'p-value': round(p, 3),
                         'Balance?': 'OK' if p > 0.05 else 'IMBALANCED'})
        except Exception:
            pass
    return pd.DataFrame(rows)

balance_df = pd.concat([
    balance_check(df_human_long, 'Human'),
    balance_check(df_llm_long,   'LLM'),
], ignore_index=True)
balance_df

,Source,Variable,AB,AC,p-value,Balance?
0,Human,age (mean),24.68,25.21,0.657,OK
1,Human,gender = Female,64.0%,75.0%,0.404,OK
2,Human,used_ai = Yes,80.0%,83.3%,0.763,OK
3,Human,heard_simplify = Yes,72.0%,62.5%,0.478,OK
4,Human,field (chi2),—,—,0.634,OK
5,Human,job_status (chi2),—,—,0.535,OK
6,Human,difficulty (chi2),—,—,0.604,OK
7,LLM,age (mean),23.66,23.43,0.074,OK
8,LLM,gender = Female,67.7%,84.9%,0.000,IMBALANCED
9,LLM,used_ai = Yes,0.0%,0.0%,1.000,OK


## Slide 4 — Average Treatment Effect (ATE)

Within-subjects paired t-test: **treatment score − control score** for each person.

In [10]:
def compute_diffs(df_long, group):
    second_page = 'B' if group == 'AB' else 'C'
    g = df_long[df_long.group == group].copy()
    a = g[g.page == 'A'].set_index('person_id')[OUTCOMES + ['gender','used_ai','heard_simplify']]
    t = g[g.page == second_page].set_index('person_id')[OUTCOMES]
    t.columns = [f'diff_{o}' for o in OUTCOMES]
    merged = a.join(t, how='inner')
    for o in OUTCOMES:
        merged[f'diff_{o}'] = merged[f'diff_{o}'] - merged[o]
    merged['group'] = group
    return merged.reset_index()

def paired_ttest(d):
    d = d.dropna().astype(float)
    n = len(d)
    if n < 3:
        return dict(n=n, mean=np.nan, se=np.nan, t=np.nan, p=np.nan, ci_lo=np.nan, ci_hi=np.nan, cohen_d=np.nan)
    mean = d.mean()
    se   = d.std(ddof=1) / np.sqrt(n)
    t, p = stats.ttest_1samp(d, popmean=0)
    lo, hi = stats.t.interval(1-ALPHA, df=n-1, loc=mean, scale=se)
    cohen_d = mean / d.std(ddof=1) if d.std(ddof=1) > 0 else np.nan
    return dict(n=n, mean=mean, se=se, t=t, p=p, ci_lo=lo, ci_hi=hi, cohen_d=cohen_d)

def ate_table(df_long, label, group):
    rows = []
    second_page = 'B' if group == 'AB' else 'C'
    diffs = compute_diffs(df_long, group)
    for o in OUTCOMES:
        r = paired_ttest(diffs[f'diff_{o}'])
        rows.append({
            'Source': label,
            'Comparison': f'Page {second_page} vs A',
            'Outcome': o,
            'n': r['n'],
            'Mean diff': round(r['mean'], 3),
            'SE': round(r['se'], 3),
            't': round(r['t'], 2) if not np.isnan(r['t']) else np.nan,
            'p': round(r['p'], 4) if not np.isnan(r['p']) else np.nan,
            '95% CI': f"[{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}]" if not np.isnan(r['ci_lo']) else '—',
            "Cohen's d": round(r['cohen_d'], 3) if not np.isnan(r['cohen_d']) else np.nan,
            'Sig.': '*' if (not np.isnan(r['p']) and r['p'] < 0.05) else '',
        })
    return pd.DataFrame(rows)

ate_df = pd.concat([
    ate_table(df_human_long, 'Human',    'AB'),
    ate_table(df_llm_long,   'LLM',      'AB'),
    ate_table(df_comb_long,  'Combined', 'AB'),
    ate_table(df_human_long, 'Human',    'AC'),
    ate_table(df_llm_long,   'LLM',      'AC'),
    ate_table(df_comb_long,  'Combined', 'AC'),
], ignore_index=True)
ate_df

,Source,Comparison,Outcome,n,Mean diff,SE,t,p,95% CI,Cohen's d,Sig.
0,Human,Page B vs A,signup,25,0.000,0.173,0.00,1.0000,"[-0.36, +0.36]",0.000,
1,Human,Page B vs A,useful,24,0.500,0.135,3.71,0.0011,"[+0.22, +0.78]",0.758,*
2,Human,Page B vs A,regular,25,0.240,0.185,1.30,0.2071,"[-0.14, +0.62]",0.259,
3,Human,Page B vs A,recommend,25,0.120,0.176,0.68,0.5025,"[-0.24, +0.48]",0.136,
4,LLM,Page B vs A,signup,533,-0.043,0.013,-3.32,0.0010,"[-0.07, -0.02]",-0.144,*
5,LLM,Page B vs A,useful,533,0.328,0.021,15.48,0.0000,"[+0.29, +0.37]",0.671,*
6,LLM,Page B vs A,regular,533,0.158,0.019,8.44,0.0000,"[+0.12, +0.19]",0.366,*
7,LLM,Page B vs A,recommend,533,0.109,0.016,6.66,0.0000,"[+0.08, +0.14]",0.288,*
8,Combined,Page B vs A,signup,558,-0.041,0.015,-2.83,0.0049,"[-0.07, -0.01]",-0.120,*
9,Combined,Page B vs A,useful,557,0.336,0.021,15.89,0.0000,"[+0.29, +0.38]",0.673,*


## Slide 5 — Heterogeneous Treatment Effects (HTE)

ATE by subgroup: gender, AI usage, Simplify awareness. Focus on the `signup` outcome (change to another outcome by editing the cell).

In [11]:
SUBGROUPS = {
    'gender':         {'Female': 'Female', 'Male': 'Male'},
    'used_ai':        {'Used AI = Yes': 'Yes', 'Used AI = No': 'No'},
    'heard_simplify': {'Heard Simplify = Yes': 'Yes', 'Heard Simplify = No/Not sure': None},
}

def hte_subgroups(df_long, group, outcome='signup', label='Combined'):
    diffs = compute_diffs(df_long, group)
    rows = []
    r = paired_ttest(diffs[f'diff_{outcome}'])
    rows.append({'Source': label, 'Group': group, 'Subgroup': 'Overall',
                 'n': r['n'], 'Mean diff': round(r['mean'],3),
                 'SE': round(r['se'],3), 'p': round(r['p'],4) if not np.isnan(r['p']) else np.nan,
                 'Sig.': '*' if (not np.isnan(r['p']) and r['p'] < 0.05) else ''})
    for col, cats in SUBGROUPS.items():
        for lbl, val in cats.items():
            if val is None:
                sub = diffs[diffs[col].isin(['No', 'Not sure'])]
            else:
                sub = diffs[diffs[col] == val]
            r = paired_ttest(sub[f'diff_{outcome}'])
            rows.append({'Source': label, 'Group': group, 'Subgroup': lbl,
                         'n': r['n'], 'Mean diff': round(r['mean'],3),
                         'SE': round(r['se'],3), 'p': round(r['p'],4) if not np.isnan(r['p']) else np.nan,
                         'Sig.': '*' if (not np.isnan(r['p']) and r['p'] < 0.05) else ''})
    return pd.DataFrame(rows)

OUTCOME = 'signup'
hte_df = pd.concat([
    hte_subgroups(df_human_long, 'AB', OUTCOME, 'Human'),
    hte_subgroups(df_llm_long,   'AB', OUTCOME, 'LLM'),
    hte_subgroups(df_comb_long,  'AB', OUTCOME, 'Combined'),
    hte_subgroups(df_human_long, 'AC', OUTCOME, 'Human'),
    hte_subgroups(df_llm_long,   'AC', OUTCOME, 'LLM'),
    hte_subgroups(df_comb_long,  'AC', OUTCOME, 'Combined'),
], ignore_index=True)
hte_df

,Source,Group,Subgroup,n,Mean diff,SE,p,Sig.
0,Human,AB,Overall,25,0.000,0.173,1.0000,
1,Human,AB,Female,16,-0.062,0.249,0.8056,
2,Human,AB,Male,9,0.111,0.200,0.5943,
3,Human,AB,Used AI = Yes,20,-0.100,0.204,0.6295,
4,Human,AB,Used AI = No,5,0.400,0.245,0.1778,
5,Human,AB,Heard Simplify = Yes,18,-0.056,0.235,0.8162,
6,Human,AB,Heard Simplify = No/Not sure,7,0.143,0.143,0.3559,
7,LLM,AB,Overall,533,-0.043,0.013,0.0010,*
8,LLM,AB,Female,361,-0.017,0.013,0.2013,
9,LLM,AB,Male,172,-0.099,0.029,0.0009,*


## Slide 6 — OLS Regression with Interactions

`diff_outcome ~ female + ai_yes + heard_yes` for each group, plus combined with `is_page_C` to compare C vs B. Change `OUTCOME` below to rerun on other outcomes.

In [13]:
def make_reg_df(df_long, group, outcome):
    d = compute_diffs(df_long, group).copy()
    d['female']    = (d['gender'] == 'Female').astype(int)
    d['ai_yes']    = (d['used_ai'] == 'Yes').astype(int)
    d['heard_yes'] = (d['heard_simplify'] == 'Yes').astype(int)
    d[f'diff_{outcome}'] = d[f'diff_{outcome}'].astype(float)
    return d.dropna(subset=[f'diff_{outcome}', 'female', 'ai_yes', 'heard_yes'])

def run_ols(df_long, outcome, label):
    ab = make_reg_df(df_long, 'AB', outcome)
    ac = make_reg_df(df_long, 'AC', outcome)
    cb = pd.concat([ab, ac], ignore_index=True)
    cb['is_page_C'] = (cb['group'] == 'AC').astype(int)

    print(f'\n========== [{label}]  outcome = {outcome} ==========')
    for name, df, formula in [
        ('B vs A', ab, f'diff_{outcome} ~ female + ai_yes + heard_yes'),
        ('C vs A', ac, f'diff_{outcome} ~ female + ai_yes + heard_yes'),
        ('C vs B (combined)', cb, f'diff_{outcome} ~ female + ai_yes + heard_yes + is_page_C'),
    ]:
        if len(df) <= 5:
            print(f'\n[{name}] too few observations ({len(df)})')
            continue
        m = smf.ols(formula, data=df).fit()
        print(f'\n[{name}]  n = {int(m.nobs)}   R² = {m.rsquared:.3f}')
        print(m.summary2().tables[1].round(4).to_string())

OUTCOME = 'signup'
for label, df in [('Human', df_human_long), ('LLM', df_llm_long), ('Combined', df_comb_long)]:
    run_ols(df, OUTCOME, label)


========== [Human]  outcome = signup ==========


NameError: name 'smf' is not defined

## Slide 7 (Report) — Power Calculations

Power is computed from the observed human effect sizes; `Needed n` = minimum sample size for 80% power at α = 0.05.

In [ ]:
def power_table(df_long, label):
    analysis = smp.TTestPower()
    rows = []
    for group in ['AB', 'AC']:
        second_page = 'B' if group == 'AB' else 'C'
        diffs = compute_diffs(df_long, group)
        for o in OUTCOMES:
            d = diffs[f'diff_{o}'].dropna().astype(float)
            n = len(d)
            if n < 3 or d.std(ddof=1) == 0:
                continue
            cohen_d = abs(d.mean() / d.std(ddof=1))
            power_now = analysis.power(effect_size=cohen_d, nobs=n, alpha=ALPHA, alternative='two-sided')
            n_needed = (int(np.ceil(analysis.solve_power(effect_size=cohen_d, power=0.80,
                                                         alpha=ALPHA, alternative='two-sided')))
                        if cohen_d > 0 else np.inf)
            rows.append({'Source': label, 'Comparison': f'Page {second_page} vs A',
                         'Outcome': o, 'n': n,
                         "Cohen's d": round(cohen_d, 3),
                         'Power @ n': f'{power_now:.1%}',
                         'Needed n (80% power)': n_needed})
    return pd.DataFrame(rows)

power_df = pd.concat([
    power_table(df_human_long, 'Human'),
    power_table(df_llm_long,   'LLM'),
    power_table(df_comb_long,  'Combined'),
], ignore_index=True)
power_df

## Save all slide tables to CSV

Optional — writes every table above to `data/slide_*.csv` so you can version them or import to other tools.

In [ ]:
overview.to_csv(os.path.join(DATA_DIR, 'slide_data_overview.csv'))
summary_df.to_csv(os.path.join(DATA_DIR, 'slide_summary_stats.csv'), index=False)
balance_df.to_csv(os.path.join(DATA_DIR, 'slide_balance_check.csv'), index=False)
ate_df.to_csv(os.path.join(DATA_DIR, 'slide_ate.csv'), index=False)
hte_df.to_csv(os.path.join(DATA_DIR, 'slide_hte.csv'), index=False)
power_df.to_csv(os.path.join(DATA_DIR, 'slide_power.csv'), index=False)
print('Saved slide_*.csv to', DATA_DIR)